In [1]:
#!/usr/bin/env python3
"""
🌾 Crop Recommendation System (with Area Code Support)
-----------------------------------------------------
✅ Loads dataset from CSV
✅ Handles 'area_code' feature correctly (encoded numerically)
✅ Crop Ranking via RandomForestClassifier
✅ Rainfall Prediction via LinearRegression
✅ Clustering with KMeans
✅ GUI Interface (Tkinter)
Author: Praneeth / Modified for Surya
"""

import tkinter as tk
from tkinter import ttk, simpledialog, messagebox
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

# ---------------- Dataset ----------------
def load_dataset(file_path):
    """Load crop dataset from CSV."""
    try:
        df = pd.read_csv(file_path)
        print(f"✅ Loaded dataset with shape {df.shape}")
        return df
    except Exception as e:
        messagebox.showerror("Error", f"Failed to load dataset:\n{e}")
        return None

# ---------------- ML Core ----------------
class CropML:
    def __init__(self, file_path):
        self.df = load_dataset(file_path)
        if self.df is None:
            raise ValueError("Dataset not loaded!")

        # Base features (including area_code if available)
        self.features = ["N", "P", "K", "temperature", "humidity", "pH", "rainfall"]
        if "area_code" in self.df.columns:
            self.features.append("area_code")

        self.target = "crop"
        self.encoder = LabelEncoder()
        self.scaler = StandardScaler()
        self.rf_model = None
        self.reg_model = None
        self.cluster_model = None
        self.area_encoder = None
        self.train_models()

    def train_models(self):
        """Train RandomForest, LinearRegression, and KMeans models."""
        # ---- RandomForest Classification ----
        X = self.df[self.features].copy()

        # Encode area_code if present
        if "area_code" in X.columns:
            le_area = LabelEncoder()
            X["area_code"] = le_area.fit_transform(X["area_code"])
            self.area_encoder = le_area
        else:
            self.area_encoder = None

        y = self.encoder.fit_transform(self.df[self.target])
        X_scaled = self.scaler.fit_transform(X)

        rf = RandomForestClassifier(n_estimators=200, random_state=42)
        rf.fit(X_scaled, y)
        self.rf_model = rf

        # ---- Linear Regression (Rainfall Prediction) ----
        X_reg = self.df[self.features[:-1]]  # exclude rainfall
        y_reg = self.df["rainfall"]
        reg = LinearRegression()
        reg.fit(X_reg, y_reg)
        self.reg_model = reg

        # ---- KMeans Clustering ----
        X2 = self.df[self.features].copy()
        if "area_code" in X2.columns and self.area_encoder is not None:
            X2["area_code"] = self.area_encoder.transform(X2["area_code"])

        X_scaled2 = self.scaler.fit_transform(X2)
        kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
        kmeans.fit(X_scaled2)
        self.cluster_model = kmeans

    def predict_all(self, input_values):
        """Predict crops, rainfall, and cluster info."""
        arr = np.array([input_values])
        arr_scaled = self.scaler.transform(arr)

        # ---- Crop Ranking ----
        probs = self.rf_model.predict_proba(arr_scaled)[0]
        crops = self.encoder.classes_
        probs = probs / probs.sum() * 100
        crop_ranking = sorted(zip(crops, probs), key=lambda x: x[1], reverse=True)
        crop_ranking = [(name, round(p, 2)) for name, p in crop_ranking[:5]]
        top_crop = crop_ranking[0][0]

        # ---- Rainfall Distribution ----
        X_reg = np.array([input_values[:-1]])
        total_rainfall = self.reg_model.predict(X_reg)[0]
        stages = ["Seedling", "Vegetative", "Flowering", "Maturity"]
        stage_rainfall = [
            round(total_rainfall * 0.2, 2),
            round(total_rainfall * 0.3, 2),
            round(total_rainfall * 0.3, 2),
            round(total_rainfall * 0.2, 2),
        ]

        # ---- Cluster Info ----
        cluster_pred = int(self.cluster_model.predict(arr_scaled)[0])
        cluster_df = self.df[self.cluster_model.labels_ == cluster_pred]
        soil_info = {
            "N_avg": round(cluster_df["N"].mean(), 2),
            "P_avg": round(cluster_df["P"].mean(), 2),
            "K_avg": round(cluster_df["K"].mean(), 2),
            "pH_avg": round(cluster_df["pH"].mean(), 2),
            "Temp_avg": round(cluster_df["temperature"].mean(), 2),
            "Humidity_avg": round(cluster_df["humidity"].mean(), 2),
        }

        return top_crop, crop_ranking, stage_rainfall, cluster_pred, soil_info


# ---------------- GUI ----------------
class CropApp:
    def __init__(self, root, file_path):
        self.root = root
        self.root.title("🌾 Crop Recommendation System")
        self.root.geometry("1200x950")
        self.ml = CropML(file_path)
        self.range_vars = {}
        self.build_ui()

    def build_ui(self):
        ttk.Label(
            self.root,
            text="🌱 Crop ML System with Ranking & Stage Rainfall",
            font=("Helvetica", 20, "bold"),
        ).pack(pady=10)

        # Input sliders
        range_frame = ttk.LabelFrame(self.root, text="Set Soil & Environment Values")
        range_frame.pack(fill=tk.X, padx=10, pady=10)

        slider_params = {
            "N": (0, 140),
            "P": (0, 90),
            "K": (0, 90),
            "temperature": (10, 40),
            "humidity": (30, 100),
            "pH": (4, 9),
            "rainfall": (0, 300),
        }

        for feature, (low, high) in slider_params.items():
            f_frame = ttk.Frame(range_frame)
            f_frame.pack(fill=tk.X, pady=4)
            ttk.Label(f_frame, text=f"{feature.capitalize():<15}", width=15).pack(side=tk.LEFT)
            var = tk.DoubleVar(value=round((low + high) / 2, 2))
            slider = ttk.Scale(
                f_frame,
                from_=low,
                to=high,
                variable=var,
                orient="horizontal",
                length=400,
                command=lambda val, var=var: var.set(round(float(val), 2)),
            )
            slider.pack(side=tk.LEFT, padx=5)
            lbl_val = ttk.Label(f_frame, textvariable=var, width=7)
            lbl_val.pack(side=tk.LEFT, padx=5)
            self.range_vars[feature] = var

        # Buttons
        btn_frame = ttk.Frame(self.root)
        btn_frame.pack(pady=10)
        ttk.Button(btn_frame, text="Predict All", command=self.predict_all).pack(side=tk.LEFT, padx=5)
        ttk.Button(btn_frame, text="Show Dataset", command=self.show_dataset).pack(side=tk.LEFT, padx=5)

        self.result_label = ttk.Label(self.root, text="Results: ", font=("Helvetica", 14))
        self.result_label.pack(pady=5)

        # Text box with scrollbar
        text_frame = ttk.Frame(self.root)
        text_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        self.text_box = tk.Text(text_frame, height=40, wrap="word", font=("Consolas", 10))
        self.text_box.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar = ttk.Scrollbar(text_frame, command=self.text_box.yview)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.text_box.configure(yscrollcommand=scrollbar.set)

    def predict_all(self):
        area_code = simpledialog.askstring("Area Code", "Enter your Area Code:")
        if not area_code:
            messagebox.showwarning("Input Required", "Please enter an Area Code!")
            return

        input_values = []
        for f in self.ml.features:
            if f == "area_code":
                if self.ml.area_encoder:
                    try:
                        area_code_num = self.ml.area_encoder.transform([area_code])[0]
                    except ValueError:
                        area_code_num = 0  # unseen area code
                else:
                    area_code_num = 0
                input_values.append(area_code_num)
            else:
                input_values.append(round(self.range_vars[f].get(), 2))

        top_crop, crop_ranking, stage_rainfall, cluster_pred, soil_info = self.ml.predict_all(input_values)

        self.result_label.config(text=f"🌱 Crop.C: {top_crop}   🔶 Cluster.K: {cluster_pred}")

        self.text_box.delete("1.0", tk.END)
        self.text_box.insert(tk.END, f"✅ Predictions Summary for Area: {area_code}\n\n")
        self.text_box.insert(tk.END, "🌱 Crop (Classification).C Ranking:\n")
        for i, (crop, prob) in enumerate(crop_ranking, 1):
            self.text_box.insert(tk.END, f" {i}. {crop}: {prob}%\n")

        self.text_box.insert(tk.END, "\n🌧 Rainfall (Regression).R per Stage:\n")
        stages = ["Seedling", "Vegetative", "Flowering", "Maturity"]
        for s, r in zip(stages, stage_rainfall):
            self.text_box.insert(tk.END, f" {s}: {r} mm\n")

        self.text_box.insert(tk.END, f"\n🔶 Cluster (Clustering).K: {cluster_pred}\n")
        self.text_box.insert(tk.END, "Soil & Environment Averages in Cluster:\n")
        for k, v in soil_info.items():
            self.text_box.insert(tk.END, f" - {k}: {v}\n")

    def show_dataset(self):
        self.text_box.delete("1.0", tk.END)
        self.text_box.insert(tk.END, "📊 Dataset (First 20 Rows):\n\n")
        self.text_box.insert(tk.END, self.ml.df.head(20).to_string(index=False))


# ---------------- Main ----------------
if __name__ == "__main__":
    root = tk.Tk()
    file_path = r"C:\Users\surya\Downloads\crop_dataset_with_area.csv"
    app = CropApp(root, file_path)
    root.mainloop()


✅ Loaded dataset with shape (500, 9)


C:\Users\surya\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1436: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
C:\Users\surya\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\surya\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
